# Notebook 1: Classical Probability

Before we can appreciate what makes quantum mechanics strange, we need a solid picture of
**classical probability** -- the kind you already know from coins, dice, and weather forecasts.

In this notebook we will:

1. Build classical probability vectors and check their properties.
2. Visualize distributions as bar charts.
3. Sample outcomes from a distribution (with a seeded RNG for reproducibility).
4. Apply a stochastic transition matrix to evolve a distribution.
5. Highlight the **key constraints** that classical probabilities obey -- constraints that quantum amplitudes will later violate.

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from quantum_demo.classical import (
    normalize_probabilities,
    sample_classical,
    apply_stochastic_matrix,
    indicator_distribution,
)

## 1. What is a probability vector?

A classical probability distribution over $n$ outcomes is a vector

$$\mathbf{p} = (p_0,\; p_1,\; \dots,\; p_{n-1})$$

satisfying two rules:

| Rule | Meaning |
|------|---------|
| **Non-negativity** | Every entry $p_i \ge 0$ |
| **Normalization** | The entries sum to 1: $\sum_i p_i = 1$ |

There is **no notion of phase or sign**. Every entry is a plain, non-negative real number.
This will be the crucial difference when we meet quantum amplitudes later.

## 2. Defining probability vectors

In [ ]:
# A fair coin: two outcomes with equal probability
fair_coin = np.array([0.5, 0.5])
print("Fair coin:", fair_coin)
print("Sum:", fair_coin.sum())

In [ ]:
# A biased three-sided die
biased_die = np.array([0.1, 0.6, 0.3])
print("Biased die:", biased_die)
print("Sum:", biased_die.sum())
print("All non-negative?", np.all(biased_die >= 0))

In [ ]:
# A deterministic (indicator) distribution: outcome 2 in a 4-outcome space
deterministic = indicator_distribution(2, 4)
print("Deterministic distribution:", deterministic)
print("Sum:", deterministic.sum())

## 3. Normalization

If we start with a vector of non-negative weights that don't yet sum to 1,
we can **normalize** it by dividing by the total.

In [ ]:
raw_weights = np.array([3.0, 1.0, 2.0, 4.0])
print("Raw weights:", raw_weights)
print("Sum before normalization:", raw_weights.sum())

normalized = normalize_probabilities(raw_weights)
print("\nNormalized:", normalized)
print("Sum after normalization:", normalized.sum())

## 4. Visualizing distributions

Bar charts are the natural way to display a probability vector.
Each bar height is the probability of the corresponding outcome.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].bar(['H', 'T'], fair_coin, color='#4C72B0', edgecolor='black')
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Fair Coin')
axes[0].set_ylabel('Probability')

axes[1].bar(['1', '2', '3'], biased_die, color='#4C72B0', edgecolor='black')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Biased Die')

axes[2].bar(['0', '1', '2', '3'], deterministic, color='#4C72B0', edgecolor='black')
axes[2].set_ylim(0, 1.05)
axes[2].set_title('Deterministic (outcome 2)')

fig.tight_layout()
plt.show()

Notice that every bar sits **at or above zero**. You will never see a negative bar in a
classical probability chart. Keep this in mind -- quantum amplitudes *can* go negative,
and that sign is what enables interference.

## 5. Sampling from a distribution

A probability distribution tells us how *likely* each outcome is. To simulate an
experiment, we **sample** from the distribution. Each call to `sample_classical`
returns one outcome index chosen according to the probabilities.

We seed the random number generator so results are reproducible.

In [ ]:
rng = np.random.default_rng(seed=42)

print("Sampling from the biased die 20 times:")
samples = [sample_classical(biased_die, rng=rng) for _ in range(20)]
print(samples)

In [ ]:
# Let's do 10,000 samples and see if the empirical frequencies match the distribution
rng = np.random.default_rng(seed=7)
n_samples = 10_000
samples = [sample_classical(biased_die, rng=rng) for _ in range(n_samples)]

# Count occurrences
counts = np.bincount(samples, minlength=len(biased_die))
empirical_freq = counts / n_samples

print("True probabilities: ", biased_die)
print("Empirical frequencies:", empirical_freq)

In [ ]:
# Side-by-side comparison
labels = ['Outcome 0', 'Outcome 1', 'Outcome 2']
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(x - width/2, biased_die, width, label='True Probability', color='#4C72B0')
ax.bar(x + width/2, empirical_freq, width, label=f'Empirical ({n_samples:,} samples)', color='#55A868')
ax.set_ylim(0, 0.8)
ax.set_ylabel('Probability')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.set_title('Sampling converges to the true distribution')
fig.tight_layout()
plt.show()

With enough samples, the empirical frequencies approach the true probabilities.
This is the **law of large numbers** at work.

## 6. Stochastic transitions

Classical systems evolve through **stochastic matrices**. A stochastic matrix $T$ has
non-negative entries, and each column sums to 1. Multiplying $T$ by a probability vector
gives a new probability vector:

$$\mathbf{p}' = T\, \mathbf{p}$$

This is the classical analogue of applying a quantum gate. But note: $T$ can only
mix and redistribute probability. It cannot create negative entries or cancellation.

In [ ]:
# A simple 2x2 stochastic matrix ("noisy channel")
# Column j describes where probability from state j goes.
T = np.array([
    [0.9, 0.3],   # prob of ending in state 0, given we start in state 0 or 1
    [0.1, 0.7],   # prob of ending in state 1
])

print("Transition matrix T:")
print(T)
print("\nColumn sums (should be 1):", T.sum(axis=0))

In [ ]:
# Start in a definite state: all probability on state 0
p_initial = np.array([1.0, 0.0])

# Apply the transition several times
p = p_initial.copy()
history = [p.copy()]
for step in range(10):
    p = apply_stochastic_matrix(p, T)
    history.append(p.copy())

print("Step 0:", history[0])
print("Step 1:", history[1])
print("Step 5:", history[5])
print("Step 10:", history[10])

In [ ]:
# Visualize how the distribution evolves
history_arr = np.array(history)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(history_arr[:, 0], 'o-', label='P(state 0)', color='#4C72B0')
ax.plot(history_arr[:, 1], 's-', label='P(state 1)', color='#C44E52')
ax.set_xlabel('Time step')
ax.set_ylabel('Probability')
ax.set_title('Stochastic evolution: probability redistribution')
ax.legend()
ax.set_ylim(0, 1.05)
fig.tight_layout()
plt.show()

The system converges to a **steady state** -- the unique distribution that is unchanged
by the transition matrix. Classical stochastic processes always mix toward equilibrium;
they can never produce the sharp interference effects we will see in quantum mechanics.

## 7. Key takeaways

| Property | Classical Probability |
|----------|----------------------|
| **Entries** | Non-negative real numbers |
| **Normalization** | Sum to 1 |
| **Sign / Phase** | None -- no negative or complex values |
| **Combination rule** | Probabilities add: $P(A \text{ or } B) = P(A) + P(B)$ for disjoint events |
| **Evolution** | Stochastic matrices (non-negative, column-sum 1) |
| **Cancellation** | Impossible -- probabilities can only grow or stay the same when paths combine |

In the next notebook, we will meet **quantum state vectors**, where entries are complex
amplitudes that *can* be negative or complex. That single change -- allowing signed
amplitudes -- is what makes quantum interference possible.